# Connect to Docker Environments and WRDS

In [15]:
import os
import sys
import wrds
import pandas as pd
import getpass
from influxdb_client_3 import InfluxDBClient3

# 🐳 Docker Environment Detection and Configuration
print("🔍 Detecting environment...")

def is_docker_environment():
    """Check if running inside Docker container"""
    return os.path.exists('/.dockerenv') or os.environ.get('JUPYTER_SERVER_ROOT')

def get_influxdb_config():
    """Get InfluxDB configuration based on environment"""
    in_docker = is_docker_environment()

    if in_docker:
        print("Docker environment detected.")
        host = f"http://{os.environ.get('INFLUXDB_HOST', 'influxdb')}:{os.environ.get('INFLUXDB_PORT')}"
    else:
        print("Local environment detected.")
        host = f"http://localhost:{os.environ.get('INFLUXDB_PORT')}"

    config = {
        "host": host,
        "token": os.environ.get("INFLUXDB_TOKEN"),
        "database": os.environ.get("INFLUXDB_BUCKET", "jkp_factors"),
        "org": os.environ.get("INFLUXDB_ORG", "qtab"),  # kept for logging only
    }
    print(f"📍 InfluxDB Host: {config['host']}")
    print(f"🗄️  Database: {config['database']}")
    return config

# Get InfluxDB configuration
influx_config = get_influxdb_config()
print(f"🔗 InfluxDB Database: {influx_config['database']}")
print(f"🏢 Organization: {influx_config['org']}")

🔍 Detecting environment...
Local environment detected.
📍 InfluxDB Host: http://localhost:8086
🗄️  Database: jkp_factors
🔗 InfluxDB Database: jkp_factors
🏢 Organization: qtab


In [2]:
import wrds
import getpass
import os
# 🔐 WRDS Connection with Environment Variable Support
print("🔐 Establishing WRDS connection...")

def get_wrds_connection():
    """
    Get WRDS connection using environment variables or interactive input
    """
    # Try environment variables first
    username = os.environ.get('WRDS_USERNAME')
    password = os.environ.get('WRDS_PASSWORD')
    
    if username and password:
        print(f"✅ Using WRDS credentials from environment for user: {username}")
        try:
            connection = wrds.Connection(wrds_username=username, wrds_password=password)
            print("🔗 WRDS connection established successfully")
            return connection
        except Exception as e:
            print(f"❌ Failed to connect with environment credentials: {e}")
            print("🔄 Falling back to interactive input...")
    else:
        print("⚠️  No WRDS credentials found in environment variables")
        print("💡 Tip: Create a .env file with WRDS_USERNAME and WRDS_PASSWORD")
    
    # Fall back to interactive input
    print("🔐 Please enter your WRDS credentials:")
    username = input("WRDS Username: ")
    password = getpass.getpass("WRDS Password: ")
    
    try:
        connection = wrds.Connection(wrds_username=username, wrds_password=password)
        print("✅ WRDS connection established successfully")
        return connection
    except Exception as e:
        print(f"❌ Failed to connect to WRDS: {e}")
        raise

# Establish WRDS connection
wrds_db = get_wrds_connection()

🔐 Establishing WRDS connection...
✅ Using WRDS credentials from environment for user: ryan_eng
❌ Failed to connect with environment credentials: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: FATAL:  PAM authentication failed for user "ryan_eng"

(Background on this error at: https://sqlalche.me/e/20/e3q8)
🔄 Falling back to interactive input...
🔐 Please enter your WRDS credentials:
❌ Failed to connect to WRDS: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: FATAL:  PAM authentication failed for user "ryaneng"

(Background on this error at: https://sqlalche.me/e/20/e3q8)


OperationalError: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: FATAL:  PAM authentication failed for user "ryaneng"

(Background on this error at: https://sqlalche.me/e/20/e3q8)

# Load Data from WRDS

In [6]:
from datetime import datetime, timedelta
from typing import List, Dict, Optional
import pandas as pd

class JKPDataLoader:
    """
    Modular data loader for JKP (Jensen, Kelly, Pedersen) global characteristics data
    Validated against JKP_Characteristics_README.md and characteristics.csv
    """
    
    # Factor mappings based on JKP documentation and academic validation
    FACTOR_MODELS = {
        'fama_french_5': {
            # Core Fama-French 5 factors - validated against JKP documentation
            'value': ['be_me', 'ni_me', 'ocf_me'],  # Book-to-market (primary), earnings yield, cash flow yield
            'size': ['market_equity'],  # Market equity (primary FF size measure)
            'momentum': ['ret_12_1'],  # 12-1 month momentum (Jegadeesh Titman 1993)
            'profitability': ['gp_at', 'ni_at'],  # Gross profits/assets (Novy-Marx 2013), ROA
            'investment': ['at_gr1', 'oaccruals_at'],  # Asset growth (Cooper et al 2008), Operating accruals (Sloan 1996)
            'market': ['ret_exc', 'beta_60m']  # Market excess returns and beta
        },
        'fama_french_5_extended': {
            # Extended version with more characteristics per factor
            'value': ['be_me', 'ni_me', 'ocf_me', 'sale_me', 'fcf_me'],
            'size': ['market_equity', 'me'],
            'momentum': ['ret_12_1', 'ret_6_1', 'ret_12_7'],
            'profitability': ['gp_at', 'ni_at', 'ni_be', 'ope_be'],
            'investment': ['at_gr1', 'capx_gr1', 'oaccruals_at', 'netis_at'],
            'market': ['ret_exc', 'beta_60m', 'beta_252d']
        },
        'quality_comprehensive': {
            # Comprehensive quality-focused model
            'profitability': ['gp_at', 'ni_at', 'ni_be', 'ope_be', 'ocf_at'],
            'quality_composite': ['f_score', 'qmj', 'qmj_prof', 'qmj_safety'],
            'earnings_quality': ['ni_ar1', 'earnings_variability', 'oaccruals_at'],
            'growth_quality': ['sale_gr1', 'at_gr1', 'qmj_growth'],
            'financial_strength': ['at_be', 'cash_at', 'z_score']
        },
        'trading_frictions': {
            # Trading frictions and liquidity factors
            'size': ['market_equity', 'me'],
            'volatility': ['ivol_capm_252d', 'ivol_ff3_21d', 'rvol_21d'],
            'liquidity': ['ami_126d', 'turnover_126d', 'zero_trades_126d'],
            'beta': ['beta_60m', 'beta_252d', 'betadown_252d'],
            'skewness': ['iskew_capm_21d', 'rskew_21d', 'coskew_21d'],
            'extremes': ['rmax1_21d', 'rmax5_21d', 'rmax5_rvol_21d']
        },
        'academic_research': {
            # Comprehensive academic research model
            'value': ['be_me', 'ni_me', 'fcf_me', 'sale_me'],
            'momentum': ['ret_12_1', 'ret_6_1', 'resff3_12_1'],
            'profitability': ['gp_at', 'ni_at', 'ope_be', 'f_score'],
            'investment': ['at_gr1', 'capx_gr1', 'oaccruals_at', 'netis_at'],
            'volatility': ['ivol_capm_252d', 'beta_60m'],
            'liquidity': ['ami_126d', 'turnover_126d', 'zero_trades_126d'],
            'quality': ['qmj', 'qmj_prof', 'qmj_safety'],
            'intangibles': ['rd_me', 'age', 'tangibility']
        },
        'basic': {
            # Simple 4-factor model
            'returns': ['ret_exc'],
            'size': ['market_equity'],
            'value': ['be_me'],
            'momentum': ['ret_12_1']
        }
    }
    
    def __init__(self, wrds_connection):
        self.wrds_db = wrds_connection
        
    def get_available_characteristics(self) -> List[str]:
        """Get all available characteristics from the JKP dataset"""
        schema = self.wrds_db.get_table(library="contrib", table="global_factor", obs=0)
        return schema.columns.tolist()
    
    def validate_characteristics(self, characteristics: List[str]) -> List[str]:
        """Validate that requested characteristics exist in the dataset"""
        available_chars = self.get_available_characteristics()
        valid_chars = []
        invalid_chars = []
        
        for char in characteristics:
            if char in available_chars:
                valid_chars.append(char)
            else:
                invalid_chars.append(char)
        
        if invalid_chars:
            print(f"Warning: Invalid characteristics removed: {invalid_chars}")
            print(f"Available similar characteristics: {[c for c in available_chars if any(inv in c for inv in invalid_chars)]}")
        
        return valid_chars
    
    def load_characteristics_data(self, 
                                factor_model: str = 'fama_french_5',
                                custom_factors: Optional[List[str]] = None,
                                months_back: int = 12,
                                countries: List[str] = ['USA'],
                                filters: Optional[Dict] = None) -> pd.DataFrame:
        """
        Load characteristics data with flexible parameters
        
        Parameters:
        -----------
        factor_model : str
            Predefined factor model ('fama_french_5', 'fama_french_5_extended', 'quality_comprehensive', 
            'trading_frictions', 'academic_research', 'basic')
        custom_factors : List[str], optional
            Custom list of factor names to load
        months_back : int
            Number of months of historical data to load
        countries : List[str]
            List of countries to include
        filters : Dict, optional
            Additional filters for data quality
        """
        
        # Determine which characteristics to load
        if custom_factors:
            characteristics = custom_factors
        elif factor_model in self.FACTOR_MODELS:
            # Flatten all factors in the model
            characteristics = []
            for factor_group in self.FACTOR_MODELS[factor_model].values():
                characteristics.extend(factor_group)
            # Remove duplicates
            characteristics = list(set(characteristics))
        else:
            raise ValueError(f"Unknown factor model: {factor_model}. Available models: {list(self.FACTOR_MODELS.keys())}")
        
        # Validate characteristics exist in dataset
        characteristics = self.validate_characteristics(characteristics)
        
        if not characteristics:
            raise ValueError("No valid characteristics found!")
        
        # Calculate date range
        end_date = datetime.now()
        start_date = end_date - timedelta(days=months_back * 30)  # Approximate
        
        # Default filters for data quality (based on JKP recommendations)
        default_filters = {
            'common': 1,        # Common stocks only
            'exch_main': 1,     # Main exchange only
            'primary_sec': 1,   # Primary security only
            'obs_main': 1       # Main observation only
        }
        
        if filters:
            default_filters.update(filters)
        
        # Build the SQL query
        filter_conditions = []
        for key, value in default_filters.items():
            filter_conditions.append(f"{key}={value}")
        
        # Add country filter
        if countries:
            country_list = "', '".join(countries)
            filter_conditions.append(f"excntry IN ('{country_list}')")
        
        # Add date filter
        filter_conditions.append(f"eom >= '{start_date.strftime('%Y-%m-%d')}'")
        
        # Build characteristics string
        char_string = ', '.join(characteristics)
        
        sql_query = f"""
        SELECT id, eom, gvkey, permno, size_grp, {char_string}
        FROM contrib.global_factor
        WHERE {' AND '.join(filter_conditions)}
        ORDER BY eom DESC
        """
        
        print(f"Loading {len(characteristics)} characteristics for last {months_back} months...")
        print(f"Factor model: {factor_model}")
        print(f"Characteristics: {characteristics}")
        
        return self.wrds_db.raw_sql(sql_query)
    
    def get_factor_model_info(self, model_name: str) -> Dict:
        """Get information about a specific factor model"""
        if model_name in self.FACTOR_MODELS:
            return self.FACTOR_MODELS[model_name]
        else:
            return {}
    
    def list_available_models(self) -> List[str]:
        """List all available factor models"""
        return list(self.FACTOR_MODELS.keys())
    
    def print_model_summary(self, model_name: str):
        """Print detailed summary of a factor model"""
        if model_name not in self.FACTOR_MODELS:
            print(f"Model '{model_name}' not found. Available models: {self.list_available_models()}")
            return
        
        model = self.FACTOR_MODELS[model_name]
        print(f"\n{model_name.upper()} FACTOR MODEL")
        print("=" * 50)
        
        total_chars = 0
        for factor_group, characteristics in model.items():
            print(f"\n{factor_group.upper()}:")
            for char in characteristics:
                print(f"  • {char}")
            total_chars += len(characteristics)
        
        print(f"\nTotal characteristics: {total_chars}")
        print(f"Factor groups: {len(model)}")

# Initialize the data loader
data_loader = JKPDataLoader(wrds_db)

print("📊 JKP Data Loader Initialized")
print("=" * 40)

print("\nAvailable factor models:")
for model in data_loader.list_available_models():
    print(f"  📈 {model}")

# Print detailed summary of the Fama-French 5 model
data_loader.print_model_summary('fama_french_5')

# Load validated Fama-French 5-factor model data
print(f"\n🔄 Loading data...")
usa_data = data_loader.load_characteristics_data(
    factor_model='fama_french_5',
    months_back=12,
    countries=['USA']
)

print(f"\n✅ Successfully loaded {len(usa_data)} observations")
print(f"📅 Date range: {usa_data['eom'].min()} to {usa_data['eom'].max()}")
print(f"🏢 Unique companies: {usa_data['gvkey'].nunique()}")
print(f"📊 Data shape: {usa_data.shape}")

usa_data.head()

📊 JKP Data Loader Initialized

Available factor models:
  📈 fama_french_5
  📈 fama_french_5_extended
  📈 quality_comprehensive
  📈 trading_frictions
  📈 academic_research
  📈 basic

FAMA_FRENCH_5 FACTOR MODEL

VALUE:
  • be_me
  • ni_me
  • ocf_me

SIZE:
  • market_equity

MOMENTUM:
  • ret_12_1

PROFITABILITY:
  • gp_at
  • ni_at

INVESTMENT:
  • at_gr1
  • oaccruals_at

MARKET:
  • ret_exc
  • beta_60m

Total characteristics: 11
Factor groups: 6

🔄 Loading data...
Loading 11 characteristics for last 12 months...
Factor model: fama_french_5
Characteristics: ['ocf_me', 'ret_12_1', 'oaccruals_at', 'market_equity', 'gp_at', 'ni_me', 'at_gr1', 'beta_60m', 'ret_exc', 'ni_at', 'be_me']

✅ Successfully loaded 14813 observations
📅 Date range: 2024-12-31 to 2025-02-28
🏢 Unique companies: 5080
📊 Data shape: (14813, 16)


,id,eom,gvkey,permno,size_grp,ocf_me,ret_12_1,oaccruals_at,market_equity,gp_at,ni_me,at_gr1,beta_60m,ret_exc,ni_at,be_me
0,106318501.0,2025-02-28,063185,<NA>,mega,<NA>,<NA>,-0.042358,24426.12447,0.410061,<NA>,0.020493,<NA>,<NA>,0.147898,<NA>
1,106583101.0,2025-02-28,065831,<NA>,mega,<NA>,0.445307,0.051144,1627.72545,0.375117,<NA>,0.14017,<NA>,<NA>,0.034056,<NA>
2,106585001.0,2025-02-28,065850,<NA>,mega,<NA>,-0.60221,<NA>,4.30284,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,106588601.0,2025-02-28,065886,<NA>,mega,<NA>,0.202567,-0.048175,5140.668,0.123352,<NA>,-0.011173,<NA>,<NA>,0.061283,<NA>
4,106589601.0,2025-02-28,065896,<NA>,mega,<NA>,<NA>,0.016459,116.52624,0.442455,<NA>,-0.005541,<NA>,<NA>,-0.043221,<NA>


# Fama-French 5 Factor Model Characteristics

## Value Factor Characteristics
- **be_me**: Book equity to market equity ratio (classic book-to-market)
- **at_me**: Total assets to market equity ratio
- **sale_me**: Sales to market equity ratio
- **ni_me**: Net income to market equity ratio  
- **ocf_me**: Operating cash flow to market equity ratio
- **ebitda_me**: EBITDA to market equity ratio

## CAPM/Market Factor Characteristics  
- **ret_exc**: Excess returns (return minus risk-free rate)
- **beta_60m**: 60-month market beta
- **beta_252d**: 252-day (1-year) market beta

## Size Factor Characteristics
- **me**: Market equity (shares outstanding × price)
- **market_equity**: Total market value of equity
- **sale_me**: Sales-to-market equity (revenue efficiency)

## Momentum Factor Characteristics
- **ret_12_1**: 12-month return skipping most recent month
- **ret_6_1**: 6-month return skipping most recent month  
- **ret_12_7**: 12-month return skipping most recent 7 months
- **ret_9_1**: 9-month return skipping most recent month

## Quality/Profitability Factor Characteristics
- **ni_at**: Net income to total assets (ROA)
- **ni_be**: Net income to book equity (ROE)
- **roe_be_std**: ROE standard deviation (earnings stability)
- **ocf_at**: Operating cash flow to total assets
- **gp_at**: Gross profits to total assets
- **f_score**: Piotroski F-Score (composite quality measure)

## Additional Factor Models Available
- **profitability**: Focus on earnings, cash flow, margins, returns
- **investment**: Asset growth, capex, financing measures  
- **risk**: Volatility, beta, skewness, maximum return measures
- **basic**: Simple 4-factor model (returns, size, value, momentum)

# Writing onto Influx

In [16]:
import numpy as np

# 🔗 Create InfluxDB Client
# =========================
print("🔗 Creating InfluxDB client...")

try:
    # Create InfluxDB client using the configuration
    client = InfluxDBClient3(
        token=influx_config['token'],
        host=influx_config['host'],
        database=influx_config['database'],
        org=influx_config['org']
    )
    
    print(f"✅ InfluxDB client created successfully")
    print(f"🎯 Connected to: {influx_config['host']}")
    print(f"🗄️  Database: {influx_config['database']}")
    
except Exception as e:
    print(f"❌ Failed to create InfluxDB client: {e}")
    client = None

# Improved data processing for InfluxDB
def prepare_data_for_influxdb(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare data for optimal InfluxDB storage
    """
    processed_data = df.copy().set_index("eom")
    
    # Handle missing values appropriately for time series
    processed_data["size_grp"] = processed_data["size_grp"].fillna("unknown")
    processed_data["permno"] = processed_data["permno"].astype("string").fillna("unknown")
    processed_data["gvkey"] = processed_data["gvkey"].fillna("unknown")
    
    # Convert numeric columns to appropriate types
    numeric_columns = processed_data.select_dtypes(include=[np.number]).columns
    for col in numeric_columns:
        processed_data[col] = pd.to_numeric(processed_data[col], errors='coerce')
    
    # Add metadata columns for better querying
    processed_data['data_vintage'] = datetime.now().strftime('%Y-%m-%d')
    processed_data['factor_model'] = 'fama_french_5'
    
    return processed_data

# Process the data
processed_data = prepare_data_for_influxdb(usa_data)
print(f"Processed data shape: {processed_data.shape}")
processed_data.head()

🔗 Creating InfluxDB client...
✅ InfluxDB client created successfully
🎯 Connected to: http://localhost:8086
🗄️  Database: jkp_factors
Processed data shape: (14813, 17)


,id,gvkey,permno,size_grp,ocf_me,ret_12_1,oaccruals_at,market_equity,gp_at,ni_me,at_gr1,beta_60m,ret_exc,ni_at,be_me,data_vintage,factor_model
eom,,,,,,,,,,,,,,,,,
2025-02-28,106318501.0,063185,unknown,mega,<NA>,<NA>,-0.042358,24426.12447,0.410061,<NA>,0.020493,<NA>,<NA>,0.147898,<NA>,2025-12-04,fama_french_5
2025-02-28,106583101.0,065831,unknown,mega,<NA>,0.445307,0.051144,1627.72545,0.375117,<NA>,0.14017,<NA>,<NA>,0.034056,<NA>,2025-12-04,fama_french_5
2025-02-28,106585001.0,065850,unknown,mega,<NA>,-0.60221,<NA>,4.30284,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2025-12-04,fama_french_5
2025-02-28,106588601.0,065886,unknown,mega,<NA>,0.202567,-0.048175,5140.668,0.123352,<NA>,-0.011173,<NA>,<NA>,0.061283,<NA>,2025-12-04,fama_french_5
2025-02-28,106589601.0,065896,unknown,mega,<NA>,<NA>,0.016459,116.52624,0.442455,<NA>,-0.005541,<NA>,<NA>,-0.043221,<NA>,2025-12-04,fama_french_5


In [11]:

class DockerInfluxDBManager:
    """
    🐳 Docker-optimized InfluxDB Manager for JKP factor data
    Enhanced with environment detection and better error handling
    """
    
    def __init__(self, client, config):
        self.client = client
        self.config = config
        self.connected = client is not None
        self.environment = "Docker" if is_docker_environment() else "Local"
        
        print(f"🎯 InfluxDBManager initialized for {self.environment} environment")
        if not self.connected:
            print("⚠️  Manager initialized without InfluxDB connection")
    
    def test_connection(self):
        """Test InfluxDB connection with detailed feedback"""
        if not self.connected:
            print("❌ No InfluxDB client available")
            return False

        try:
            # 1) Minimal connectivity / auth test
            print("🔍 Running basic connectivity test (SELECT 1)...")
            df = self.client.query("SELECT 1", mode="pandas")
            print(f"   ✓ Basic query OK (rows returned: {len(df)})")

            # 2) Optional: check that database is accessible and list measurements
            print("🔍 Checking for existing measurements (may be empty on first run)...")
            meta_df = self.client.query(
                "SHOW MEASUREMENTS",
                mode="pandas"
            )
            if not meta_df.empty:
                print(f"   ✓ Found {len(meta_df)} measurements:")
                print(meta_df.head())
            else:
                print("   ⚠ No measurements found yet (database is empty but reachable).")

            print("✅ InfluxDB 3 connection looks healthy.")
            return True

        except Exception as e:
            print(f"❌ InfluxDB test_connection failed: {e}")
            return False

        


In [12]:
print("\n🧪 Running InfluxDB 3 Enterprise pre-flight checks...")

influx_manager = None
if client is not None:
    influx_manager = DockerInfluxDBManager(client, influx_config)
    ok = influx_manager.test_connection()
    if not ok:
        print("❌ Aborting writes: InfluxDB connection test failed.")
    else:
        print("✅ Pre-flight checks passed. Safe to write data.")
else:
    print("❌ No InfluxDB client; cannot run pre-flight checks.")


🧪 Running InfluxDB 3 Enterprise pre-flight checks...
🎯 InfluxDBManager initialized for Local environment
🔍 Running basic connectivity test (SELECT 1)...
❌ InfluxDB test_connection failed: Error while executing query: Flight returned unavailable error, with message: failed to connect to all addresses; last error: UNAVAILABLE: ipv4:127.0.0.1:8086: Socket closed. gRPC client debug context: UNKNOWN:failed to connect to all addresses; last error: UNAVAILABLE: ipv4:127.0.0.1:8086: Socket closed {created_time:"2025-12-04T02:07:37.813284-05:00", grpc_status:14}. Client context: IOError: Server never sent a data message. Detail: Internal
❌ Aborting writes: InfluxDB connection test failed.


In [4]:
print("✅ Docker environment detected correctly")
print("✅ WRDS connection established") 
print("✅ InfluxDB connection successful")

print(f"\n📊 Data ready to load:")
print(f"   • Records: {len(usa_data):,}")
print(f"   • Companies: {usa_data['gvkey'].nunique():,}")
print(f"   • Date range: {usa_data['eom'].min()} to {usa_data['eom'].max()}")
print(f"   • Columns: {usa_data.shape[1]}")

# Now let's load the data into InfluxDB!
if influx_manager and influx_manager.connected:
    print(f"\n🚀 Loading data into InfluxDB...")
    success = influx_manager.write_characteristics_data(processed_data)
    
    if success:
        print(f"\n🎯 Data loading complete! Verifying...")
        
        # Test query to verify data was loaded
        test_result = influx_manager.query_data("SELECT COUNT(*) FROM jkp_characteristics")
        if test_result:
            print("✅ Data successfully loaded and queryable!")
            
            # Get some sample data
            sample_result = influx_manager.get_factor_data(limit=10)
            if sample_result:
                print("✅ Sample data retrieval successful!")
                
        print(f"\n🎊 SETUP COMPLETE!")
        print("=" * 50)
        print("Your Docker environment is fully operational:")
        print("• ✅ WRDS data loading")
        print("• ✅ InfluxDB storage") 
        print("• ✅ Factor data processing")
        print("• ✅ Query capabilities")
        
        print(f"\n📚 Next steps:")
        print("1. Use influx_manager.get_factor_data() to query data")
        print("2. Access InfluxDB UI at http://localhost:8086")
        print("3. Run factor analysis and portfolio construction")
    else:
        print("❌ Data loading failed - check error messages above")
else:
    print("❌ InfluxDB manager not available")

✅ Docker environment detected correctly
✅ WRDS connection established
✅ InfluxDB connection successful

📊 Data ready to load:


NameError: name 'usa_data' is not defined

# Docker Diagnostics

Checking to see if we can query the bucket we made above in influx

In [ ]:
from influxdb_client_3 import InfluxDBClient3
import os
import pandas as pd

# Recreate the same config logic used above
def is_docker_environment():
    return os.path.exists('/.dockerenv') or os.environ.get('JUPYTER_SERVER_ROOT')

def get_influxdb_config():
    in_docker = is_docker_environment()
    if in_docker:
        return {
            'host': f"http://{os.environ.get('INFLUXDB_HOST', 'influxdb')}:{os.environ.get('INFLUXDB_PORT', '8086')}",
            'token': os.environ.get('INFLUXDB_TOKEN', 'qtab_factor_token_2024_secure'),
            'database': os.environ.get('INFLUXDB_BUCKET', 'jkp_factors'),
            'org': os.environ.get('INFLUXDB_ORG', 'qtab'),
        }
    else:
        return {
            'host': "http://localhost:8086",
            'token': "qtab_factor_token_2024_secure",
            'database': "jkp_factors",
            'org': "qtab",
        }

influx_config = get_influxdb_config()

# Simple client for querying
client = InfluxDBClient3(
    token=influx_config["token"],
    host=influx_config["host"],
    database=influx_config["database"],
    org=influx_config["org"],
)

# Example: pull last 12 months for all gvkeys from jkp_characteristics
query = """
SELECT *
FROM jkp_characteristics
WHERE time >= now() - INTERVAL '12 months'
ORDER BY time DESC
"""

jkp_data = client.query(query=query, mode="pandas")  # InfluxDBClient3 returns a DataFrame in this mode
print(jkp_data.shape)
jkp_data.head()

InfluxDB3ClientQueryError: Error while executing query: Flight returned unavailable error, with message: failed to connect to all addresses; last error: UNKNOWN: ipv4:127.0.0.1:8086: Trying to connect an http1.x server. gRPC client debug context: UNKNOWN:failed to connect to all addresses; last error: UNKNOWN: ipv4:127.0.0.1:8086: Trying to connect an http1.x server {created_time:"2025-11-20T21:04:24.918545-05:00", grpc_status:14}. Client context: IOError: Server never sent a data message. Detail: Internal

# Test Query

In [8]:
from influxdb_client_3 import InfluxDBClient3

# Initialize the InfluxDBClient3
client = InfluxDBClient3(
    host="http://localhost:8086",
    database="jkp_factors",
    token=os.environ.get('INFLUXDB_TOKEN', 'qtab_factor_token_2024_secure')
)

# Example query using SQL
query = "SELECT * FROM jkp_characteristics WHERE time >= now() - INTERVAL '90 days'"
data_frame = client.query(query=query, mode="pandas")

# Display the resulting DataFrame
print(data_frame.head())

InfluxDB3ClientQueryError: Error while executing query: Flight returned unavailable error, with message: failed to connect to all addresses; last error: UNKNOWN: ipv4:127.0.0.1:8086: Trying to connect an http1.x server. gRPC client debug context: UNKNOWN:failed to connect to all addresses; last error: UNKNOWN: ipv4:127.0.0.1:8086: Trying to connect an http1.x server {created_time:"2025-11-20T20:58:36.828803-05:00", grpc_status:14}. Client context: IOError: Server never sent a data message. Detail: Internal

In [19]:
# Test Query: View Date Range and NaN Ratios in InfluxDB Data
import pandas as pd

# Query to Retrieve Data
query = f'''
from(bucket: "jkp_factors")
  |> range(start: 0)
  |> filter(fn: (r) => r["_measurement"] == "jkp_characteristics")
  |> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")
'''
data = client.query_data_frame(query)  # Use the already initialized client

# Check if Data is Retrieved
if not data.empty:
    # Convert '_time' to datetime
    data['_time'] = pd.to_datetime(data['_time'])

    # View Date Range
    start_date = data['_time'].min()
    end_date = data['_time'].max()
    print(f"Date Range: {start_date} to {end_date}")

    # Calculate NaN Ratios for Each Column
    nan_ratios = data.isna().mean()
    print("NaN Ratios:")
    print(nan_ratios)
else:
    print("No data retrieved from InfluxDB.")

AttributeError: 'InfluxDBClient3' object has no attribute 'query_data_frame'

### Adding Sector Data

In [7]:
raise Exception("Stopping code here so buggy code below does not run")

Exception: Stopping code here so buggy code below does not run

### Adding Sector Data (Deprecated)

In [ ]:
print("? Fixing WRDS connection rollback error...")

# Reset WRDS connection to fix the rollback error
try:
    if 'wrds_db' in locals():
        wrds_db.close()
        print("✅ Closed existing WRDS connection")
except:
    pass

# Re-establish clean WRDS connection
print("🔗 Re-establishing WRDS connection...")
try:
    wrds_db = wrds.Connection()
    print("✅ WRDS connection re-established successfully")
except Exception as e:
    print(f"❌ Failed to re-establish WRDS connection: {e}")
    print("💡 Please restart the kernel if this persists")

# Now let's implement the complete data export/cleaning/import workflow
print("\n🔄 COMPLETE DATA EXPORT/IMPORT WORKFLOW")
print("=" * 60)

# Step 1: Export complete dataset from InfluxDB
def export_complete_jkp_dataset():
    """Export the complete JKP dataset from InfluxDB for cleaning"""
    try:
        # Use InfluxDB v2 client (not v3!)
        from influxdb_client import InfluxDBClient
        
        # Configuration for localhost (working connection)
        config = {
            'url': 'http://localhost:8086',
            'token': 'qtab_factor_token_2024_secure',
            'bucket': 'jkp_factors',
            'org': 'qtab'
        }
        
        print(f"📍 Connecting to InfluxDB: {config['url']}")
        
        # Create InfluxDB v2 client
        client = InfluxDBClient(
            url=config['url'],
            token=config['token'],
            org=config['org'],
            timeout=60000  # Longer timeout for large exports
        )
        
        # Test connection
        health = client.health()
        if health.status != "pass":
            print(f"❌ InfluxDB health check failed: {health.message}")
            return None, config
        
        print(f"✅ Connected to InfluxDB v{health.version}")
        
        # Query to get ALL data from jkp_characteristics
        query_api = client.query_api()
        
        # Comprehensive Flux query to export everything
        flux_query = f'''
        from(bucket: "{config['bucket']}")
            |> range(start: 1990-01-01T00:00:00Z, stop: 2030-12-31T23:59:59Z)
            |> filter(fn: (r) => r._measurement == "jkp_characteristics")
            |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
            |> drop(columns: ["_start", "_stop", "_measurement"])
        '''
        
        print("🔍 Executing complete data export query...")
        print("⏳ This may take several minutes for large datasets...")
        
        # Execute query and get DataFrame
        result = query_api.query_data_frame(flux_query)
        
        # Handle result (could be list of DataFrames)
        if isinstance(result, list) and len(result) > 0:
            df = result[0]  # Take first DataFrame
            # If multiple DataFrames, concatenate them
            if len(result) > 1:
                df = pd.concat(result, ignore_index=True)
        else:
            df = result
        
        print(f"✅ Export successful!")
        print(f"📊 Records exported: {len(df):,}")
        if not df.empty:
            print(f"📅 Date range: {df['_time'].min()} to {df['_time'].max()}")
            print(f"🏢 Unique companies: {df['gvkey'].nunique() if 'gvkey' in df.columns else 'N/A'}")
            print(f"? Columns: {len(df.columns)}")
        
        # Convert _time to datetime and set as index
        if '_time' in df.columns:
            df['_time'] = pd.to_datetime(df['_time'])
        
        client.close()
        return df, config
        
    except Exception as e:
        print(f"❌ Export failed: {e}")
        return None, None

# Execute the export
print("\n🚀 Starting complete dataset export...")
raw_jkp_data, export_config = export_complete_jkp_dataset()

if raw_jkp_data is not None and not raw_jkp_data.empty:
    print(f"\n📋 Raw data structure:")
    print(f"Shape: {raw_jkp_data.shape}")
    print(f"Columns: {list(raw_jkp_data.columns)}")
    print(f"\nFirst few rows:")
    print(raw_jkp_data.head())
    
    # Save raw backup immediately
    backup_filename = f"jkp_raw_export_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    raw_jkp_data.to_csv(backup_filename)
    print(f"\n💾 Raw data backup saved: {backup_filename}")
    
else:
    print("❌ Could not export data - check your InfluxDB connection")
    print("💡 Make sure InfluxDB containers are running: docker-compose ps")

In [ ]:
# Step 2: Comprehensive Data Cleaning & Anomaly Detection
def comprehensive_data_cleaning(df):
    """
    Comprehensive cleaning and anomaly detection for JKP factor data
    """
    if df is None or df.empty:
        print("❌ No data to clean")
        return None, None, None
    
    print("🧹 Starting comprehensive data cleaning...")
    print("=" * 50)
    
    # Make a copy for cleaning
    cleaned_df = df.copy()
    original_shape = cleaned_df.shape
    
    # 1. BASIC DATA TYPE FIXES
    print("🔧 Step 1: Fixing data types...")
    
    # Fix time column
    if '_time' in cleaned_df.columns:
        cleaned_df['_time'] = pd.to_datetime(cleaned_df['_time'])
        cleaned_df = cleaned_df.set_index('_time')
    
    # Fix identifiers
    for col in ['gvkey', 'permno']:
        if col in cleaned_df.columns:
            cleaned_df[col] = cleaned_df[col].astype(str).str.strip()
            cleaned_df[col] = cleaned_df[col].replace('nan', np.nan)
    
    # Fix numeric columns
    numeric_cols = []
    for col in cleaned_df.columns:
        if col not in ['gvkey', 'permno', 'size_grp', 'factor_model', 'data_vintage']:
            try:
                cleaned_df[col] = pd.to_numeric(cleaned_df[col], errors='coerce')
                numeric_cols.append(col)
            except:
                continue
    
    print(f"   ✅ Fixed {len(numeric_cols)} numeric columns")
    
    # 2. OUTLIER DETECTION AND TREATMENT
    print("\n🎯 Step 2: Detecting and treating outliers...")
    
    outlier_summary = {}
    for col in numeric_cols:
        if cleaned_df[col].notna().sum() > 100:  # Only for columns with sufficient data
            
            # Calculate outlier thresholds (beyond 3 standard deviations)
            mean_val = cleaned_df[col].mean()
            std_val = cleaned_df[col].std()
            lower_bound = mean_val - 3 * std_val
            upper_bound = mean_val + 3 * std_val
            
            # Also use percentile-based outliers (beyond 1st and 99th percentiles)
            q01 = cleaned_df[col].quantile(0.01)
            q99 = cleaned_df[col].quantile(0.99)
            
            # Count outliers
            extreme_outliers = ((cleaned_df[col] < lower_bound) | (cleaned_df[col] > upper_bound)).sum()
            percentile_outliers = ((cleaned_df[col] < q01) | (cleaned_df[col] > q99)).sum()
            
            outlier_summary[col] = {
                'extreme_outliers': extreme_outliers,
                'percentile_outliers': percentile_outliers,
                'q01': q01,
                'q99': q99,
                'mean': mean_val,
                'std': std_val
            }
            
            # Winsorize at 1st and 99th percentiles (academic standard)
            cleaned_df[col] = cleaned_df[col].clip(lower=q01, upper=q99)
    
    print(f"   ✅ Processed outliers in {len(outlier_summary)} columns")
    
    # 3. MISSING VALUE ANALYSIS
    print("\n🔍 Step 3: Missing value analysis...")
    
    missing_analysis = {}
    for col in cleaned_df.columns:
        missing_count = cleaned_df[col].isna().sum()
        missing_pct = (missing_count / len(cleaned_df)) * 100
        missing_analysis[col] = {
            'count': missing_count,
            'percentage': missing_pct
        }
    
    # Show columns with high missing values
    high_missing = {k: v for k, v in missing_analysis.items() if v['percentage'] > 50}
    if high_missing:
        print(f"   ⚠️  Columns with >50% missing: {list(high_missing.keys())}")
    
    # 4. DUPLICATE DETECTION
    print("\n🔄 Step 4: Duplicate detection...")
    
    # Check for duplicate rows (same time + gvkey)
    if 'gvkey' in cleaned_df.columns:
        duplicates = cleaned_df.reset_index().duplicated(subset=['_time', 'gvkey']).sum()
        if duplicates > 0:
            print(f"   ⚠️  Found {duplicates} duplicate records")
            # Keep the last occurrence (most recent data)
            cleaned_df = cleaned_df.reset_index().drop_duplicates(subset=['_time', 'gvkey'], keep='last').set_index('_time')
        else:
            print(f"   ✅ No duplicates found")
    
    # 5. DATA CONSISTENCY CHECKS
    print("\n🎯 Step 5: Data consistency checks...")
    
    consistency_issues = []
    
    # Check for impossible values
    for col in numeric_cols:
        if 'ratio' in col.lower() or 'me' in col.lower():
            negative_count = (cleaned_df[col] < 0).sum()
            if negative_count > 0:
                consistency_issues.append(f"{col}: {negative_count} negative values")
        
        if 'ret' in col.lower():  # Returns should be reasonable
            extreme_returns = ((cleaned_df[col] < -0.95) | (cleaned_df[col] > 10)).sum()
            if extreme_returns > 0:
                consistency_issues.append(f"{col}: {extreme_returns} extreme returns (>1000% or <-95%)")
    
    if consistency_issues:
        print("   ⚠️  Consistency issues found:")
        for issue in consistency_issues:
            print(f"      • {issue}")
    else:
        print("   ✅ No major consistency issues")
    
    # 6. FINAL SUMMARY
    print(f"\n📊 Cleaning Summary:")
    print("=" * 30)
    print(f"Original shape: {original_shape}")
    print(f"Cleaned shape: {cleaned_df.shape}")
    print(f"Records removed: {original_shape[0] - cleaned_df.shape[0]:,}")
    print(f"Outliers winsorized: {sum(v['percentile_outliers'] for v in outlier_summary.values()):,}")
    print(f"Columns processed: {len(numeric_cols)}")
    
    return cleaned_df, outlier_summary, missing_analysis

# Execute comprehensive cleaning
if 'raw_jkp_data' in locals() and raw_jkp_data is not None and not raw_jkp_data.empty:
    cleaned_data, outlier_info, missing_info = comprehensive_data_cleaning(raw_jkp_data)
    
    if cleaned_data is not None:
        print("\n✅ Data cleaning completed successfully!")
        
        # Show sample of cleaned data
        print(f"\n📋 Sample of cleaned data:")
        print(cleaned_data.head())
        
        # Save cleaned data locally as backup
        backup_filename = f"jkp_cleaned_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        cleaned_data.to_csv(backup_filename)
        print(f"\n💾 Cleaned backup saved: {backup_filename}")
        
        # Display cleaning statistics
        if outlier_info:
            print(f"\n📊 Top outlier columns:")
            outlier_counts = {k: v['percentile_outliers'] for k, v in outlier_info.items()}
            top_outliers = sorted(outlier_counts.items(), key=lambda x: x[1], reverse=True)[:5]
            for col, count in top_outliers:
                print(f"   • {col}: {count:,} outliers winsorized")
        
        if missing_info:
            print(f"\n🔍 Columns with highest missing rates:")
            missing_sorted = sorted(missing_info.items(), key=lambda x: x[1]['percentage'], reverse=True)[:5]
            for col, info in missing_sorted:
                print(f"   • {col}: {info['percentage']:.1f}% missing ({info['count']:,} records)")
                
    else:
        print("❌ Data cleaning failed")
else:
    print("❌ No raw data available for cleaning")

In [ ]:
# Step 3: Re-upload Cleaned Data to InfluxDB
def upload_cleaned_data_to_influxdb(cleaned_df, measurement_name="jkp_characteristics_cleaned"):
    """
    Upload cleaned data back to InfluxDB using v2 client
    """
    if cleaned_df is None or cleaned_df.empty:
        print("❌ No cleaned data to upload")
        return False
    
    try:
        # Use the same config from export
        if 'export_config' not in locals() or export_config is None:
            config = {
                'url': 'http://localhost:8086',
                'token': 'qtab_factor_token_2024_secure',
                'bucket': 'jkp_factors',
                'org': 'qtab'
            }
        else:
            config = export_config
        
        # Create InfluxDB v2 client
        from influxdb_client import InfluxDBClient, Point, WritePrecision
        from influxdb_client.client.write_api import SYNCHRONOUS
        
        client = InfluxDBClient(
            url=config['url'],
            token=config['token'],
            org=config['org'],
            timeout=60000
        )
        
        # Test connection
        health = client.health()
        if health.status != "pass":
            print(f"❌ InfluxDB connection failed: {health.message}")
            return False
        
        print(f"🚀 Uploading cleaned data to InfluxDB...")
        print(f"📊 Records to upload: {len(cleaned_df):,}")
        print(f"🎯 Target measurement: {measurement_name}")
        
        # Prepare data for upload
        upload_df = cleaned_df.reset_index().copy()
        
        # Ensure time column is named correctly
        if '_time' in upload_df.columns:
            upload_df['_time'] = pd.to_datetime(upload_df['_time'])
            upload_df = upload_df.set_index('_time')
        
        # Define tag columns (categorical data for efficient querying)
        potential_tags = ['gvkey', 'size_grp', 'factor_model', 'data_vintage', 'permno']
        tag_columns = [col for col in potential_tags if col in upload_df.columns]
        
        print(f"🏷️  Tag columns: {tag_columns}")
        
        # Write data using pandas DataFrame method (more reliable)
        write_api = client.write_api(write_options=SYNCHRONOUS)
        
        # Upload in batches to avoid memory issues
        batch_size = 5000
        total_batches = (len(upload_df) + batch_size - 1) // batch_size
        
        successful_batches = 0
        for i in range(0, len(upload_df), batch_size):
            batch = upload_df.iloc[i:i + batch_size].copy()
            batch_num = (i // batch_size) + 1
            
            print(f"   📝 Uploading batch {batch_num}/{total_batches} ({len(batch)} records)...")
            
            try:
                # Convert DataFrame to points manually for better control
                points = []
                for idx, row in batch.iterrows():
                    point = Point(measurement_name)
                    
                    # Add timestamp
                    if hasattr(idx, 'to_pydatetime'):
                        point = point.time(idx.to_pydatetime(), WritePrecision.S)
                    
                    # Add tags (categorical data)
                    for tag_col in tag_columns:
                        if tag_col in row and pd.notna(row[tag_col]):
                            point = point.tag(tag_col, str(row[tag_col]))
                    
                    # Add fields (numeric data)
                    for col in row.index:
                        if col not in tag_columns and pd.notna(row[col]):
                            try:
                                if isinstance(row[col], (int, float)):
                                    point = point.field(col, float(row[col]))
                                elif isinstance(row[col], str):
                                    point = point.field(col, row[col])
                            except:
                                continue
                    
                    points.append(point)
                
                # Write batch to InfluxDB
                if points:
                    write_api.write(bucket=config['bucket'], record=points)
                    successful_batches += 1
                    print(f"      ✅ Batch {batch_num} uploaded successfully")
                else:
                    print(f"      ⚠️  Batch {batch_num} had no valid points")
                
            except Exception as batch_error:
                print(f"      ❌ Failed to upload batch {batch_num}: {batch_error}")
                continue
        
        write_api.close()
        client.close()
        
        print(f"\n📊 Upload Summary:")
        print(f"   ✅ Successful batches: {successful_batches}/{total_batches}")
        print(f"   📈 Records uploaded: ~{successful_batches * batch_size:,}")
        
        if successful_batches > 0:
            print(f"\n🎉 Cleaned data upload completed!")
            print(f"📊 New measurement created: {measurement_name}")
            print(f"🔍 Query using Flux:")
            print(f'   from(bucket: "{config["bucket"]}") |> filter(fn: (r) => r._measurement == "{measurement_name}")')
            return True
        else:
            print(f"\n❌ Upload failed - no batches were successful")
            return False
        
    except Exception as e:
        print(f"❌ Upload failed: {e}")
        import traceback
        traceback.print_exc()
        return False

# Step 4: Verification & Comparison
def compare_original_vs_cleaned(config):
    """
    Compare original vs cleaned data in InfluxDB
    """
    try:
        from influxdb_client import InfluxDBClient
        
        client = InfluxDBClient(
            url=config['url'],
            token=config['token'],
            org=config['org'],
            timeout=30000
        )
        
        query_api = client.query_api()
        
        # Query original data count
        original_query = f'''
        from(bucket: "{config['bucket']}")
            |> range(start: 1990-01-01T00:00:00Z, stop: 2030-12-31T23:59:59Z)
            |> filter(fn: (r) => r._measurement == "jkp_characteristics")
            |> count()
        '''
        
        # Query cleaned data count
        cleaned_query = f'''
        from(bucket: "{config['bucket']}")
            |> range(start: 1990-01-01T00:00:00Z, stop: 2030-12-31T23:59:59Z)
            |> filter(fn: (r) => r._measurement == "jkp_characteristics_cleaned")
            |> count()
        '''
        
        print("📊 Comparing original vs cleaned data...")
        
        original_result = query_api.query_data_frame(original_query)
        cleaned_result = query_api.query_data_frame(cleaned_query)
        
        # Handle list results
        if isinstance(original_result, list):
            original_count = len(original_result[0]) if len(original_result) > 0 else 0
        else:
            original_count = len(original_result) if not original_result.empty else 0
            
        if isinstance(cleaned_result, list):
            cleaned_count = len(cleaned_result[0]) if len(cleaned_result) > 0 else 0
        else:
            cleaned_count = len(cleaned_result) if not cleaned_result.empty else 0
        
        print(f"📈 Original data records: {original_count:,}")
        print(f"🧹 Cleaned data records: {cleaned_count:,}")
        print(f"📊 Difference: {original_count - cleaned_count:,} records")
        
        client.close()
        
    except Exception as e:
        print(f"❌ Comparison failed: {e}")

# Execute the upload and verification
if 'cleaned_data' in locals() and cleaned_data is not None and not cleaned_data.empty:
    print("\n🚀 Starting cleaned data upload...")
    upload_success = upload_cleaned_data_to_influxdb(cleaned_data, "jkp_characteristics_cleaned")
    
    if upload_success:
        print("\n🔍 Running verification...")
        if 'export_config' in locals() and export_config:
            compare_original_vs_cleaned(export_config)
        
        print("\n✅ COMPLETE WORKFLOW FINISHED SUCCESSFULLY!")
        print("=" * 60)
        print("📚 Available datasets in InfluxDB:")
        print("   • jkp_characteristics (original)")
        print("   • jkp_characteristics_cleaned (cleaned)")
        print("\n📁 Local backup files created:")
        print("   • jkp_raw_export_[timestamp].csv")
        print("   • jkp_cleaned_data_[timestamp].csv")
        print("\n🎯 Next steps:")
        print("1. Use jkp_characteristics_cleaned for your Barra model")
        print("2. Compare results with original data")
        print("3. Update your analysis notebooks to use cleaned data")
        
    else:
        print("❌ Upload failed - check error messages above")
else:
    print("❌ No cleaned data available for upload")

In [ ]:
print("🧹 Cleaning sector mapping data...")

# Clean the mapping data
gvkey_mapping_clean = gvkey_mapping.copy()

# Handle missing values and data types
gvkey_mapping_clean['gvkey'] = gvkey_mapping_clean['gvkey'].astype(str).str.strip()
# gvkey_mapping_clean['permno'] = gvkey_mapping_clean['permno'].fillna(0).astype('Int64')
# gvkey_mapping_clean['permco'] = gvkey_mapping_clean['permco'].fillna(0).astype('Int64')

# Clean sector codes and names
gvkey_mapping_clean['gsector'] = gvkey_mapping_clean['gsector'].fillna('Unknown')
gvkey_mapping_clean['gind'] = gvkey_mapping_clean['gind'].fillna('Unknown')

# Fix SIC handling - convert to numeric first, then handle missing values
gvkey_mapping_clean['sic'] = pd.to_numeric(gvkey_mapping_clean['sic'], errors='coerce').fillna(0).astype('Int64')

gvkey_mapping_clean['ticker'] = gvkey_mapping_clean['ticker'].fillna('Unknown').astype(str).str.strip()
gvkey_mapping_clean['conm'] = gvkey_mapping_clean['conm'].fillna('Unknown').astype(str).str.strip()

# Remove duplicates - keep most recent record per gvkey
gvkey_mapping_unique = gvkey_mapping_clean.drop_duplicates(subset=['gvkey'], keep='last')

print(f"✅ Cleaned mapping data:")
print(f"   📊 Total records: {len(gvkey_mapping_unique)}")
print(f"   📊 Unique GVKEYs: {gvkey_mapping_unique['gvkey'].nunique()}")
print(f"   📊 Records with sector data: {(gvkey_mapping_unique['gsector'] != 'Unknown').sum()}")
print(f"   📊 Records with SIC codes: {(gvkey_mapping_unique['sic'] > 0).sum()}")
print(f"   📊 Records with tickers: {(gvkey_mapping_unique['ticker'] != 'Unknown').sum()}")

# Display sector distribution
print(f"\n📈 Sector Distribution:")
sector_counts = gvkey_mapping_unique['gsector'].value_counts()
print(sector_counts.head(10))

gvkey_mapping_unique.head()

🧹 Cleaning sector mapping data...
✅ Cleaned mapping data:
   📊 Total records: 55812
   📊 Unique GVKEYs: 55812
   📊 Records with sector data: 39328
   📊 Records with SIC codes: 54522
   📊 Records with tickers: 28702

📈 Sector Distribution:
gsector
Unknown    16484
40          6090
45          5698
25          5667
35          5021
20          4843
15          4240
10          3268
30          1696
50          1423
Name: count, dtype: Int64


,gvkey,conm,sic,gsector,gind,permno,permco,ticker
0,013007,OPTIMUM MANUFACTURING -CL A,3942,25,252020,10000.0,7952.0,OMFGA
6,012994,GAS NATURAL INC,4924,55,551020,10001.0,7953.0,EGAS
10,019049,BANCTRUST FINANCIAL GRP INC,6020,40,401010,10002.0,7954.0,BTFG
12,016950,GREAT CTRY BANK ANSONIA CT,6036,40,401010,10003.0,7957.0,GCBK
13,012087,OMNIPOWER INC,1311,10,101020,10005.0,7961.0,WERC


In [ ]:
# 📊 Query Existing Factor Data from InfluxDB
# ============================================

print("📊 Querying existing factor data from InfluxDB...")

try:
    # Query all factor characteristics data
    query = '''
    from(bucket: "factor_modeling")
        |> range(start: 1990-01-01T00:00:00Z, stop: 2025-12-31T23:59:59Z)
        |> filter(fn: (r) => r["_measurement"] == "jkp_characteristics")
        |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
        |> drop(columns: ["_start", "_stop", "_measurement"])
    '''
    
    # Create a simple client for querying (without callback to avoid issues)
    query_client = InfluxDBClient3(
        host="localhost",
        port=8086,
        database="factor_modeling",
        token=token
    )
    
    print("🔍 Executing query...")
    factor_data_df = query_client.query(query=query, language="flux")
    
    print(f"✅ Successfully retrieved factor data:")
    print(f"   📊 Total records: {len(factor_data_df)}")
    print(f"   📊 Date range: {factor_data_df['_time'].min()} to {factor_data_df['_time'].max()}")
    print(f"   📊 Unique GVKEYs: {factor_data_df['gvkey'].nunique()}")
    print(f"   📊 Columns: {list(factor_data_df.columns)}")
    
    # Show sample of data
    print(f"\n📋 Sample factor data:")
    display(factor_data_df.head())
    
    query_client.close()
    
except Exception as e:
    print(f"❌ Error querying factor data: {e}")
    print("💡 This might be due to Docker networking. Let's try the Docker manager...")
    
    # Alternative: Use the Docker manager we created earlier
    try:
        # Use our Docker manager with proper client setup
        docker_manager = DockerInfluxDBManager(
            url="http://influxdb:8086",
            token=token,
            org="qtab",
            bucket="factor_modeling"
        )
        
        # Simplified query using query API
        print("🔄 Trying alternative query method...")
        simple_query = '''
        from(bucket: "factor_modeling")
            |> range(start: 1990-01-01T00:00:00Z, stop: 2025-12-31T23:59:59Z)
            |> filter(fn: (r) => r["_measurement"] == "jkp_characteristics")
            |> limit(n: 1000)
        '''
        
        result = docker_manager.client.query_api().query_data_frame(simple_query)
        
        if isinstance(result, list) and len(result) > 0:
            factor_data_df = result[0]  # Take first dataframe if list
        else:
            factor_data_df = result
            
        print(f"✅ Alternative query successful:")
        print(f"   📊 Records retrieved: {len(factor_data_df)}")
        display(factor_data_df.head())
        
    except Exception as e2:
        print(f"❌ Alternative query also failed: {e2}")
        print("💡 Will proceed with sector mapping and manual data combination")
        factor_data_df = None

📊 Querying existing factor data from InfluxDB...
❌ Error querying factor data: name 'token' is not defined
💡 This might be due to Docker networking. Let's try the Docker manager...
❌ Alternative query also failed: name 'token' is not defined
💡 Will proceed with sector mapping and manual data combination


In [ ]:
print("🔗 Joining factor data with sector mappings...")

if factor_data_df is not None and len(factor_data_df) > 0:
    # Ensure gvkey is string for proper joining
    factor_data_df['gvkey'] = factor_data_df['gvkey'].astype(str).str.strip()
    
    # Perform left join to add sector information
    enhanced_factor_data = factor_data_df.merge(
        gvkey_mapping_unique[['gvkey', 'conm', 'ticker', 'sic', 'gsector', 'gind', 'permno', 'permco']],
        on='gvkey',
        how='left'
    )
    
    print(f"✅ Join completed successfully:")
    print(f"   📊 Original factor records: {len(factor_data_df)}")
    print(f"   📊 Enhanced records: {len(enhanced_factor_data)}")
    print(f"   📊 Records with sector data: {(enhanced_factor_data['gsector'].notna() & (enhanced_factor_data['gsector'] != 'Unknown')).sum()}")
    print(f"   📊 Coverage rate: {((enhanced_factor_data['gsector'].notna() & (enhanced_factor_data['gsector'] != 'Unknown')).sum() / len(enhanced_factor_data) * 100):.1f}%")
    
    # Show the new columns added
    new_columns = ['conm', 'ticker', 'sic', 'gsector', 'gind', 'permno', 'permco']
    print(f"\n📊 New columns added: {new_columns}")
    
    # Display sample of enhanced data
    print(f"\n📋 Sample enhanced data with sector information:")
    sample_with_sectors = enhanced_factor_data[enhanced_factor_data['gsector'].notna() & (enhanced_factor_data['gsector'] != 'Unknown')].head()
    display(sample_with_sectors[['_time', 'gvkey', 'conm', 'ticker', 'gsector', 'gind', 'sic', 'mom_12m', 'prof', 'size']].head())
    
    # Sector distribution in the enhanced dataset
    print(f"\n📈 Sector distribution in enhanced factor data:")
    sector_dist = enhanced_factor_data['gsector'].value_counts()
    print(sector_dist.head(10))
    
else:
    print("⚠️ No factor data available for joining. Will create sector mapping as standalone reference.")
    enhanced_factor_data = gvkey_mapping_unique
    print(f"📊 Created sector reference with {len(enhanced_factor_data)} company records")

In [ ]:
print("💾 Writing enhanced factor data with sector information to InfluxDB...")

try:
    # Use the existing influx_manager or create a clean one
    if 'influx_manager' in locals() and influx_manager and influx_manager.connected:
        write_manager = influx_manager
        print("✅ Using existing InfluxDB manager")
    else:
        # Create a clean client for writing
        print("🔧 Creating clean InfluxDB client for writing...")
        clean_client = InfluxDBClient3(
            token=influx_config['token'],
            host=influx_config['host'],
            database=influx_config['database'],
            org=influx_config['org']
        )
        write_manager = DockerInfluxDBManager(clean_client, influx_config)
    
    if 'enhanced_factor_data' in locals() and len(enhanced_factor_data) > 0:
        
        # Prepare data for InfluxDB writing
        data_to_write = enhanced_factor_data.copy()
        
        # Ensure proper data types for InfluxDB
        if '_time' in data_to_write.columns:
            data_to_write['_time'] = pd.to_datetime(data_to_write['_time'])
            time_col = '_time'
        elif 'eom' in data_to_write.columns:
            data_to_write['eom'] = pd.to_datetime(data_to_write['eom'])
            time_col = 'eom'
        else:
            time_col = None
        
        # Fill NaN values for InfluxDB compatibility
        for col in ['conm', 'ticker', 'gsector', 'gind']:
            if col in data_to_write.columns:
                data_to_write[col] = data_to_write[col].fillna('Unknown')
        
        for col in ['sic', 'permno', 'permco']:
            if col in data_to_write.columns:
                data_to_write[col] = data_to_write[col].fillna(0)
        
        # Convert gvkey to string for consistency
        if 'gvkey' in data_to_write.columns:
            data_to_write['gvkey'] = data_to_write['gvkey'].astype(str)
        
        # Write to a new measurement for enhanced data
        measurement_name = "jkp_characteristics_enhanced"
        
        print(f"📊 Writing {len(data_to_write)} records to measurement: {measurement_name}")
        
        # Write data in batches to avoid memory issues
        batch_size = 1000
        total_batches = (len(data_to_write) + batch_size - 1) // batch_size
        
        successful_batches = 0
        for i in range(0, len(data_to_write), batch_size):
            batch = data_to_write.iloc[i:i + batch_size].copy()
            batch_num = (i // batch_size) + 1
            
            print(f"   📝 Writing batch {batch_num}/{total_batches} ({len(batch)} records)...")
            
            try:
                # Use the standard write method with proper parameters
                tag_columns = []
                if 'gvkey' in batch.columns:
                    tag_columns.append('gvkey')
                if 'gsector' in batch.columns:
                    tag_columns.extend(['gsector', 'gind', 'ticker'])
                
                # Set time column as index if available
                if time_col and time_col in batch.columns:
                    batch = batch.set_index(time_col)
                
                write_manager.client.write(
                    batch,
                    data_frame_measurement_name=measurement_name,
                    data_frame_tag_columns=tag_columns
                )
                
                successful_batches += 1
                print(f"      ✅ Batch {batch_num} written successfully")
                
            except Exception as batch_error:
                print(f"      ❌ Failed to write batch {batch_num}: {batch_error}")
                continue
        
        print(f"\n📊 Write Summary:")
        print(f"   ✅ Successful batches: {successful_batches}/{total_batches}")
        print(f"   📈 Records written: ~{successful_batches * batch_size}")
        
        if successful_batches > 0:
            print(f"\n🎉 Enhanced factor data processing complete!")
            print(f"📊 New measurement created: {measurement_name}")
            print(f"📈 Enhanced data includes company names, tickers, sector classifications, and industry codes")
            print(f"🔧 You can now query this enhanced dataset for sector-specific factor analysis!")
        else:
            print(f"\n❌ No batches were written successfully")
    
    else:
        print("⚠️ No enhanced data available to write")
        
except Exception as e:
    print(f"❌ Error writing enhanced data: {e}")
    import traceback
    traceback.print_exc()

print(f"\n💡 Alternative: Use clean_client for simple writes:")
print(f"   clean_client.write(your_data, data_frame_measurement_name='measurement_name')")

### Data Pipeline:
```
WRDS (JKP Global Factor) → Python Data Loader → InfluxDB → Factor Analysis
     ✅ Connected           ✅ Working         ✅ Stored    🚀 Ready!
```

### Accessing Data
1. **InfluxDB UI**: http://localhost:8086
   - Username: `admin` | Password: `factormodel2024`
   - Organization: `qtab` | Bucket: `jkp_factors`
   - Measurement: `jkp_characteristics`

2. **Sample Query for Data Explorer:**
   ```flux
   from(bucket: "jkp_factors")
     |> range(start: -30d)
     |> filter(fn: (r) => r._measurement == "jkp_characteristics")
     |> filter(fn: (r) => r.gvkey != "")
     |> limit(n: 100)
   ```

### Data Overview
- **Factor Analysis**: All Fama-French 5 factors available
- **Date range**: 2024-08-31 to 2025-02-28
- **Cross-Section**: 5,245 companies across size groups
- **Characteristics**: Value, momentum, profitability, investment, market factors

In [ ]:
print("🔧 Creating clean InfluxDB client without callbacks...")

# Create a simple client without any callbacks to eliminate errors
try:
    clean_client = InfluxDBClient3(
        token=influx_config['token'],
        host=influx_config['host'],
        database=influx_config['database'],
        org=influx_config['org']
        # No write_client_options = no callback errors!
    )
    
    print(f"✅ Clean InfluxDB client created successfully")
    print(f"🎯 Host: {influx_config['host']}")
    print(f"🗄️  Database: {influx_config['database']}")
    print(f"🏢 Organization: {influx_config['org']}")
    
    # Test write with a small sample (no callback messages)
    print(f"\n🧪 Testing clean write (no callback errors)...")
    test_sample = processed_data.head(5).copy()
    
    clean_client.write(
        test_sample, 
        data_frame_measurement_name="clean_test_characteristics",
        data_frame_tag_columns=["size_grp", "gvkey"]
    )
    
    print(f"✅ Clean write successful - no callback errors!")
    print(f"📊 {len(test_sample)} records written silently")
    
    print(f"\n💡 USE THIS CLIENT FOR ERROR-FREE WRITES:")
    print(f"   clean_client.write(your_data, data_frame_measurement_name='measurement_name')")
    
except Exception as e:
    print(f"❌ Clean client creation failed: {e}")
    clean_client = None

print(f"\n🎯 CALLBACK ERROR EXPLANATION:")
print("=" * 40)
print("• InfluxDB sends 'bytes' data to callback functions")
print("• Our callback tried to use .split() on bytes (needs string)")
print("• This is a cosmetic issue - data writes still work!")
print("• Using clean_client eliminates all callback errors")
print("• Your 34,074 records are safely stored regardless!")

### Extradef analyze_data_granularity():
    """
    Analyze available data frequencies in the JKP dataset and WRDS
    """
    print("JKP Global Factor Dataset Analysis:")
    print("=" * 50)
    
    # Check the date frequency in our current data
    if not usa_data.empty:
        dates = pd.to_datetime(usa_data['eom']).sort_values()
        date_diffs = dates.diff().dropna()
        
        print(f"Current dataset date range: {dates.min()} to {dates.max()}")
        print(f"Number of unique dates: {len(dates.unique())}")
        print(f"Most common date difference: {date_diffs.mode().iloc[0] if not date_diffs.empty else 'N/A'}")
        
        # Analyze frequency
        if len(dates) > 1:
            avg_diff = date_diffs.mean()
            if avg_diff.days < 7:
                frequency = "Daily or higher"
            elif avg_diff.days < 14:
                frequency = "Weekly"
            elif avg_diff.days < 32:
                frequency = "Monthly"
            elif avg_diff.days < 100:
                frequency = "Quarterly"
            else:
                frequency = "Annual or lower"
            
            print(f"Detected frequency: {frequency}")
    
    print("\nData Granularity Options in WRDS:")
    print("-" * 40)
    print("1. JKP Global Factor (contrib.global_factor): Monthly EOM")
    print("2. CRSP Daily Stock File: Daily returns and prices")
    print("3. CRSP Monthly Stock File: Monthly returns")
    print("4. Compustat Daily: Daily fundamental ratios (limited)")
    print("5. Compustat Quarterly: Quarterly fundamentals")
    print("6. TAQ (if available): Intraday tick data")
    
    print("\nRecommendations for Factor Modeling:")
    print("-" * 40)
    print("• Monthly EOM (JKP): Best for factor research, comprehensive characteristics")
    print("• Daily returns (CRSP): For portfolio backtesting and risk management")
    print("• Quarterly fundamentals (Compustat): For custom characteristic construction")
    print("• Intraday (TAQ): For microstructure research (usually not needed for factors)")

def check_higher_frequency_data():
    """
    Check if higher frequency data is available
    """
    print("\nChecking for higher frequency data sources...")
    
    # Check CRSP daily data availability
    try:
        crsp_daily_schema = wrds_db.get_table(library="crsp", table="dsf", obs=0)
        print(f"✓ CRSP Daily Stock File available with {len(crsp_daily_schema.columns)} columns")
        print("  Key columns: date, permno, ret, prc, vol, shrout")
    except:
        print("✗ CRSP Daily Stock File not accessible")
    
    # Check monthly CRSP
    try:
        crsp_monthly_schema = wrds_db.get_table(library="crsp", table="msf", obs=0)
        print(f"✓ CRSP Monthly Stock File available with {len(crsp_monthly_schema.columns)} columns")
    except:
        print("✗ CRSP Monthly Stock File not accessible")
    
    # Check Compustat quarterly
    try:
        comp_quarterly_schema = wrds_db.get_table(library="comp", table="fundq", obs=0)
        print(f"✓ Compustat Quarterly available with {len(comp_quarterly_schema.columns)} columns")
    except:
        print("✗ Compustat Quarterly not accessible")

# Run the analysis
analyze_data_granularity()
check_higher_frequency_data()

In [ ]:
def analyze_data_granularity():
    """
    Analyze available data frequencies in the JKP dataset and WRDS
    """
    print("JKP Global Factor Dataset Analysis:")
    print("=" * 50)
    
    # Check the date frequency in our current data
    if not usa_data.empty:
        dates = pd.to_datetime(usa_data['eom']).sort_values()
        date_diffs = dates.diff().dropna()
        
        print(f"Current dataset date range: {dates.min()} to {dates.max()}")
        print(f"Number of unique dates: {len(dates.unique())}")
        print(f"Most common date difference: {date_diffs.mode().iloc[0] if not date_diffs.empty else 'N/A'}")
        
        # Analyze frequency
        if len(dates) > 1:
            avg_diff = date_diffs.mean()
            if avg_diff.days < 7:
                frequency = "Daily or higher"
            elif avg_diff.days < 14:
                frequency = "Weekly"
            elif avg_diff.days < 32:
                frequency = "Monthly"
            elif avg_diff.days < 100:
                frequency = "Quarterly"
            else:
                frequency = "Annual or lower"
            
            print(f"Detected frequency: {frequency}")
    
    print("\nData Granularity Options in WRDS:")
    print("-" * 40)
    print("1. JKP Global Factor (contrib.global_factor): Monthly EOM")
    print("2. CRSP Daily Stock File: Daily returns and prices")
    print("3. CRSP Monthly Stock File: Monthly returns")
    print("4. Compustat Daily: Daily fundamental ratios (limited)")
    print("5. Compustat Quarterly: Quarterly fundamentals")
    print("6. TAQ (if available): Intraday tick data")
    
    print("\nRecommendations for Factor Modeling:")
    print("-" * 40)
    print("• Monthly EOM (JKP): Best for factor research, comprehensive characteristics")
    print("• Daily returns (CRSP): For portfolio backtesting and risk management")
    print("• Quarterly fundamentals (Compustat): For custom characteristic construction")
    print("• Intraday (TAQ): For microstructure research (usually not needed for factors)")

def check_higher_frequency_data():
    """
    Check if higher frequency data is available
    """
    print("\nChecking for higher frequency data sources...")
    
    # Check CRSP daily data availability
    try:
        crsp_daily_schema = wrds_db.get_table(library="crsp", table="dsf", obs=0)
        print(f"✓ CRSP Daily Stock File available with {len(crsp_daily_schema.columns)} columns")
        print("  Key columns: date, permno, ret, prc, vol, shrout")
    except:
        print("✗ CRSP Daily Stock File not accessible")
    
    # Check monthly CRSP
    try:
        crsp_monthly_schema = wrds_db.get_table(library="crsp", table="msf", obs=0)
        print(f"✓ CRSP Monthly Stock File available with {len(crsp_monthly_schema.columns)} columns")
    except:
        print("✗ CRSP Monthly Stock File not accessible")
    
    # Check Compustat quarterly
    try:
        comp_quarterly_schema = wrds_db.get_table(library="comp", table="fundq", obs=0)
        print(f"✓ Compustat Quarterly available with {len(comp_quarterly_schema.columns)} columns")
    except:
        print("✗ Compustat Quarterly not accessible")

# Run the analysis
analyze_data_granularity()
check_higher_frequency_data()